<a href="https://colab.research.google.com/github/muskanjaved584/edge-mobility-anomaly-detector/blob/main/01_baseline_training_and_onnx_ipynb_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q ultralytics onnx onnxruntime-gpu

from ultralytics import YOLO

# 1. Load the lightweight YOLO11 nano model
model = YOLO("yolo11n.pt")

# 2. Train on the sample dataset (~30 seconds)
results = model.train(
    data="coco8.yaml",
    epochs=25,
    imgsz=640,
    batch=16,
    name="ajou_mobility_baseline"
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco8.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0

In [3]:
# Export trained model to edge-optimized ONNX FP16 format
onnx_path = model.export(format="onnx", half=True, simplify=True)
print(f"Exported edge model saved at: {onnx_path}")

# Benchmark PyTorch vs ONNX inference speed
pt_metrics = model.val(data="coco8.yaml")
onnx_model = YOLO(onnx_path)
onnx_metrics = onnx_model.val(data="coco8.yaml")

print(f"PyTorch Speed: {pt_metrics.speed['inference']:.2f} ms/frame")
print(f"ONNX FP16 Speed: {onnx_metrics.speed['inference']:.2f} ms/frame")


WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO11n summary (fused): 100 layers, 2,616,248 parameters, 0 gradients, 6.5 GFLOPs

PyTorch: starting from '/content/runs/detect/ajou_mobility_baseline/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (5.3 MB)
requirements: Ultralytics requirement ['onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 10 packages in 382ms
Prepared 2 packages in 36ms
Installed 2 packages in 5ms
 + colorama==0.4.6
 + onnxslim==0.1.99

requirements: AutoUpdate success ✅ 0.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.23.2 o